# **Deep Q-Networks (DQN) [Assignment]**

<hr>

### **Part 6**: Reinforcement Learning (from Zero to One)

*African Institute for Mathematical Sciences (AIMS), South Africa
9 March, 2026*

**Arnu Pretorius** - Research Scientist, InstaDeep

*Credits*: Adapted from Deep Learning Indaba 2022. Apache License 2.0.


## Setup

In [1]:
# @title Install required packages (run me) { display-mode: "form" }
# @markdown This may take a minute or two to complete.
%%capture
!sudo apt install swig
!pip install jaxlib
!pip install jax
!pip install git+https://github.com/deepmind/dm-haiku
!pip install gymnasium
!pip install gymnasium[box2d]
!pip install optax
!pip install matplotlib
!pip install chex

In [2]:
# @title Import required packages (run me) { display-mode: "form" }
%%capture
import copy
from shutil import rmtree # deleting directories
import random
import collections # useful data structures
import numpy as np
import gymnasium as gym # reinforcement learning environments
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp # jax numpy
import haiku as hk # jax neural network library
import optax # jax optimizer library
import matplotlib.pyplot as plt # graph plotting library
from IPython.display import HTML
from base64 import b64encode
import chex

# Hide warnings
import warnings
warnings.filterwarnings('ignore')
np.bool8 = np.bool_

### **Environment (warm-up)**

We will begin by using the simple **CartPole** environment. In CartPole, the task is for the agent to learn to balance a pole for as long as possible by moving a cart *left* or *right*.

<img src="https://miro.medium.com/max/600/1*v8KcdjfVGf39yvTpXDTCGQ.gif" width="30%" />

- **State space**: The state of the environment is represented by four numbers; *angular position of the pole, angular velocity of the pole, position of the cart, velocity of the cart*.
- **Action space**: There are only two actions; *left* and *right*. As such, the actions can be represented by integers $0$ and $1$.  
- **Dynamics**: State transitions are deterministic and the agent receives a reward of `1` for every timestep the pole is still upright. If the pole falls over, the game is over and the agent receives no more reward. The game is also over after `500` timesteps, so the maximum reward the agent can collect is `500`.

In CartPole, the environment is considered solved when the agent can reliably achieve an episode return of 500.

In [3]:
# Create the environment
env_name = "LunarLander-v3"  # "LunarLander-v2" (for later...)
env = gym.make(env_name)

# Reset the environment
s_0, _ = env.reset()
print("Initial State::", s_0)

# Get environment obs space
obs_shape = env.observation_space.shape
print("Environment Obs Space Shape:", obs_shape)

# Get action space - e.g. discrete or continuous
print(f"Environment action space: {env.action_space}")

# Get num actions
num_actions = env.action_space.n
print(f"Number of actions: {num_actions}")

Initial State:: [-0.00708303  1.3996296  -0.7174529  -0.50181794  0.00821429  0.16251376
  0.          0.        ]
Environment Obs Space Shape: (8,)
Environment action space: Discrete(4)
Number of actions: 4


## **Q-Learning**


In Q-learning the agent learns a function that approximates the **value** of state-action pairs. By *value* we mean the return you expect to receive if you start in a particular state $S_t$, take a particular action $A_t$, and then act according to a particular policy $\pi$ forever after. The state-action value function of policy $\pi$ is given by

$Q_\pi(s,a)=\mathrm{E}_{\pi}\left[G_t \mid S_t=s,\ A_t=a\right]$.

We say that the value function $Q_\pi(s,a)$ is the **optimal** value function if the policy $\pi$ is an optimal policy. We denote the optimal value function as follows:

$Q_\ast(s,a)=\max \limits_\pi \  \mathrm{E}_{\pi}\left[G_t \mid S_t=s,\ A_t=a\right]$

There is an important relationship between the optimal action $a_\ast$ in a state $s$ and the optimal state-action value function $Q_\ast$. Namely, the optimal action $a_\ast$ in state $s$ is equal to the action that maximises the optimal state-action value function. This relationship naturally induces an optimal policy:

$\pi_\ast(s)=\arg \max \limits_a\ Q_\ast(s, a)$

### **Greedy action selection**

---
> **For you!**
>
> Implement a greedy action selector. This should be a function that, given a vector of Q-values, returns the action with the largest Q-value.
---

**Useful methods:**
*   `jnp.argmax` ([docs](https://jax.readthedocs.io/en/latest/_autosummary/jax.numpy.argmax.html)) [note from above we had `import jax.numpy as jnp`]

In [4]:
# Implement a function takes q-values as input and returns the greedy_action
def select_greedy_action(q_values):

  # YOUR CODE
  action = jnp.argmax(q_values)
  # END YOUR CODE

  return action

In [5]:
# @title Check your implementation (run me) {display-mode: "form"}

try:
  q_values = jnp.array([1,1,3,4])
  action = select_greedy_action(q_values)

  if action != 3:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except Exception as e:
  print("Oops! It seems you didn't implement anything?")


Looks good!


### **Q-Network**
In Q-learning, we parametrise the Q-function using a neural network $Q_\phi$. We obtain a policy from the Q-network by always choosing the action with the *greatest* value:

$\hat{\pi}_\phi(s)=\arg \max \limits_a\ Q_{\phi}(s, a)$

We use a neural network to approximate this Q-function. The network will take an observation as input and then output a Q-value for each of the available actions. So in the case of CartPole, the output of the network will have size $2$.

In [6]:
def build_network(num_actions: int, layers=[20, 20]) -> hk.Transformed:
  """Factory for a simple MLP network for approximating Q-values."""

  def q_network(obs):
    network = hk.Sequential(
        [hk.Flatten(),
         hk.nets.MLP(layers + [num_actions])])
    return network(obs)

  return hk.without_apply_rng(hk.transform(q_network))

Let's initialise our Q-network and get the initial parameters.

In [7]:
# Initialise Q-network
Q_NETWORK = build_network(num_actions=num_actions, layers=[128, 128]) # two actions

dummy_obs = jnp.zeros((1,*obs_shape), jnp.float32) # a dummy observation like the one in CartPole

random_key = jax.random.PRNGKey(42) # random key
Q_NETWORK_PARAMS = Q_NETWORK.init(random_key, dummy_obs) # Get initial params

print("Q-Learning params:", Q_NETWORK_PARAMS.keys())

Q-Learning params: dict_keys(['mlp/~/linear_0', 'mlp/~/linear_1', 'mlp/~/linear_2'])


### **The Bellman Equations**
The value function can be written recursively as:

$Q_{\pi}(s, a) = \mathbb{E}_{s^\prime \sim p(s^\prime |s, a)}\left[r + \gamma\underset{a^{\prime} \sim \pi}{\mathrm{E}}\left[Q_{\pi}\left(s^{\prime}, a^{\prime}\right)\right]\right]$,

Intuitively, this equation says that the value of the action $a$ you took in the state $s$ is equal to the reward $r$ you expect to get, plus the value you expect to get in the next state $s'$ you land in given that you will choose your next action $a'$ with the policy $\pi$. The Bellman equation for the optimal value function is:

$Q_{*}(s, a) = \mathbb{E}_{s^\prime \sim p(s^\prime |s, a)}\left[r +\ \gamma \underset{a^{\prime}}{\max}\ Q_{*}(s^{\prime}, a^{\prime})\right]$

Notice that instead of chosing your next action $a'$ with policy $\pi$, we choose the action with the greatest Q-value.

### **The Bellman Backup**
To learn to approximate the optimal Q-value function, we can use the right-hand side of the Bellman equation as an update rule. In other words, suppose we have a Q-network $Q_\phi$, with parameters $\phi$, then we can iteratively update the parameters such that

$Q_\phi(s,a)\leftarrow r + \gamma \underset{a'}{\max}\ Q_\phi(s', a')$.

Intuitively, this says that the approximation of the Q-value of action $a$ in state $s$ should be updated such that it is closer to being equal to the reward received from the environment $r$ plus the value of the best possible action in the next state $s'$. We can perform this optimisation by minimising the difference between the left and right-hand side, with respect to the parameters $\phi$ using gradient descent. We can measure the difference between the two values using the [squared-error](https://en.wikipedia.org/wiki/Mean_squared_error#Loss_function).

---
> **For you!**
>
> Implement the squared-error function.
---

**Useful functions**
* `jnp.square` ([docs](https://jax.readthedocs.io/en/latest/_autosummary/jax.numpy.square.html))

In [8]:
def compute_squared_error(pred, target):
  # YOUR CODE
  squared_error = jnp.mean((pred - target) ** 2)
  # END YOUR CODE

  return squared_error

In [9]:
#@title Check your implementation (run me) {display-mode: "form"}

try:
  result = compute_squared_error(1, 4)

  if result != 9:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except Exception as e:
  print("Oops! It seems you didn't implement anything?")

Looks good!




---
> **For you!**
>
> Implement a function that computes the **Bellman target** (right-hand side of the Bellman equation). If the episode is at the last timestep (i.e. done==1.0), then the Bellman target should be equal to the reward, with no extra value at the end.
---

**Useful functions**
* `jnp.max` ([docs](https://jax.readthedocs.io/en/latest/_autosummary/jax.numpy.max.html))

In [10]:
# Bellman target
def compute_bellman_target(reward, done, next_q_values, gamma=0.99):
  """A function to compute the bellman target.

  Args:
      reward: a scalar reward.
      done: a scalar of value either 1.0 or 0.0, indicating if the transition is a terminal one.
      next_q_values: a vector of q_values for the next state. One for each action.
      gamma: a scalar discount value between 0 and 1.

    -- IMPORTANT NOTE: you should keep the default gamma value to check your implementation

  Returns:
      A scalar equal to the bellman target.

  """
  # YOUR CODE
  bellman_target = reward + (1.0 - done) * gamma * jnp.max(next_q_values)
  # END YOUR CODE

  return bellman_target

In [11]:
#@title Check your implementation (run me) {display-mode: "form"}

try:
  # not done
  result1 = compute_bellman_target(1, 0.0, np.array([3,2], "float32"))

  # done
  result2 = compute_bellman_target(1, 1.0, np.array([3,2], "float32"))

  if np.abs(result1 - 3.97) > 0.0001 or result2 != 1:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except Exception as e:
  print("Oops! It seems you didn't implement anything?")

Looks good!


We can now combine these two functions to compute the loss for Q-learning. The Q-learning loss is equal to the squared difference between the predicted Q-value of an action and its corresponding Bellman target.

---
> **For you!**
>
> Implement the Q-learning loss.
---



In [12]:
def q_learning_loss(q_values, action, reward, done, next_q_values):
    """Implementation of the Q-learning loss.T

    Args:
        q_values: a vector of Q-values, one for each action.
        action: an integer, giving the action that was chosen. q_values[action] is the value of the chose action.
        done: is a scalar that indicates if this is a terminal transition.
        next_q_values: a vector of Q-values in the next state.
    Returns:
        The squared difference between the q_value of the chosen action and the bellman target.
    """
    # YOUR CODE
    chosen_action_q_value = q_values[action]  # q_value of action, use array indexing
    bellman_target = compute_bellman_target(reward, done, next_q_values)
    squared_error = compute_squared_error(chosen_action_q_value, bellman_target)
    # END YOUR CODE

    return squared_error

In [13]:
#@title Check your implementation (run me) {display-mode: "form"}

try:
  result = q_learning_loss(np.array([3,2], "float32"), 1, 2, 0.0, np.array([3,2], "float32"))
  print(result)

  if np.abs(result - 8.820902) > 0.0001:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except Exception as e:
  print("Oops! It seems you didn't implement anything?")

8.820902
Looks good!


### **Target Q-network**
Notice that when we compute the bellman target we are using our Q-network $Q_\phi$ to compute the value for the next state $S_{t+1}$. We are basically using our latest approximation of the Q-function to compute the target of our next approximation. Using an approximation to compute the target for your next approximation, is called **bootstrapping**.

Unfortunately, if we naively bootstrap like this, it can make training a neural network very unstable. To mitigage this we can instead use a different set of parameters $\bar{\phi}$ to compute the values at state $S_{t+1}$. We will keep the parameters $\bar{\phi}$ fixed and only periodically update them to be equal to the latest online parameters $\phi$ every couple of training steps *(say 100)*. This serves to keep the bellman targets fixed for a couple training steps to help reduce the instability due to bootstrapping.


We will need to keep track of the latest (online) parameters $\phi$, as well as the target networks parameters $\bar{\phi}$. Lets make a `NamedTuple` to store these two values. We will also need to keep track of the number of learner steps we have taken, so that we know when to update the target network. Lets store a `count` of the learn steps in the `learn_state`.

In [14]:
# Store online and target parameters
QLearnParams = collections.namedtuple("Params", ["online", "target"])

# Q-learn-state
QLearnState = collections.namedtuple("LearnerState", ["count", "optim_state"])

We will be using **Optax** to optimize our neural network. We store the state of the optimizer in the `learn_state` above. Lets now instantiate the optimizer and add the initial Q-network parameters to a `QLearnParams` object.

In [15]:
# Initialise Q-network optimizer
# Q_LEARN_OPTIMIZER = optax.adam(1e-6) # learning rate

Q_LEARN_OPTIMIZER = optax.chain(
    optax.clip_by_global_norm(0.5),  # clip gradients
    optax.adam(1e-4) # learning rate
)

Q_LEARN_OPTIM_STATE = Q_LEARN_OPTIMIZER.init(Q_NETWORK_PARAMS) # initial optim state

# Create Learn State
Q_LEARNING_LEARN_STATE = QLearnState(0, Q_LEARN_OPTIM_STATE) # count set to zero initially

# Add initial Q-network weights to QLearnParams object
Q_LEARNING_PARAMS = QLearnParams(online=Q_NETWORK_PARAMS, target=Q_NETWORK_PARAMS) # target equal to online

Now we can implement a simple function that updates the target network parameters to equal the latest online network parameters every 100 training steps.

In [16]:
def update_target_params(learn_state, online_weights, target_weights, update_frequency=100):
  """A function to update target params every 100 training steps"""

  target = jax.lax.cond(
      jnp.mod(learn_state.count, update_frequency) == 0,
      lambda x, y: x,
      lambda x, y: y,
      online_weights,
      target_weights
  )

  params = QLearnParams(online_weights, target)

  return params

### **Q-learning loss**
We now have everything we need to implement the `q_learn` function which takes some batch of transitions and does a step of Q-learning to update the network paramters. But first we use `jax.vmap` to modify the `q_learning_loss` function so that it accepts batches of transitions. In addition, we will compute the Q-values by passing the observations through the `Q_NETWORK` and the target Q-values using the target parameters of the `Q_NETWORK`.

In [17]:
def batched_q_learning_loss(online_params, target_params, obs, actions, rewards, next_obs, dones):
    q_values = Q_NETWORK.apply(online_params, obs) # use the online parameters
    next_q_values = Q_NETWORK.apply(target_params, next_obs) # use the target parameters
    squared_error = jax.vmap(q_learning_loss)(q_values, actions, rewards, dones, next_q_values) # vmap q_learning_loss
    mean_squared_error = jnp.mean(squared_error) # mean squared error over batch
    return mean_squared_error

Now we can create the `q_learn` function which computes the gradient of the `batched_q_learning_loss` and then uses an Optax optimizer to update the network weights and then finally (maybe) updates the target parameters.

In [18]:
def q_learn(rng, params, learner_state, memory):
  # Compute gradients
  grad_loss = jax.grad(batched_q_learning_loss)(params.online, params.target, memory.obs,
                                          memory.action, memory.reward,
                                          memory.next_obs, memory.done,
                                          ) # jax.grad

  # Get updates
  updates, opt_state = Q_LEARN_OPTIMIZER.update(grad_loss, learner_state.optim_state)

  # Apply them
  new_weights = optax.apply_updates(params.online, updates)

  # Maybe update target network
  params = update_target_params(learner_state, new_weights, params.target)

  # Increment learner step counter
  learner_state = QLearnState(learner_state.count + 1, opt_state)

  return params, learner_state

### **A General Purpose RL Training Loop**
We have implemented a general purpose RL training loop for you. The training loop takes several arguments as input but the three most important for you to understand are `agent_select_action_func`, `agent_learn_func` and the `agent_memory`.

* The `agent_select_action_func` should take an observation and set of `agent_params` as input and should return an action.
* The `agent_learn_func` should take the agent's parameters and some "memories" as input and then update and return the agents new parameters.
* The `agent_memory` is a general purpose module we define that can store some relevant information about the agent's experiences in the environment that can be used in the `agent_learn_func`.

Below is the training loop function. You are welcome to go through the code and try to understand it but this is not strictly required.

In [19]:
#@title Training loop (run me) { display-mode: "form" }

# NamedTuple to store transitions
Transition = collections.namedtuple("Transition", ["obs", "action", "reward", "next_obs", "done"])

# Training Loop
def run_training_loop(env_name, agent_params, agent_select_action_func,
    agent_actor_state=None, agent_learn_func=None, agent_learner_state=None,
    agent_memory=None, num_episodes=1000, evaluator_period=10,
    evaluation_episodes=8, learn_steps_per_episode=1,
    train_every_timestep=False, video_subdir="",):
    """
    This function runs several episodes in an environment and periodically does
    some agent learning and evaluation.

    Args:
        env: a gym environment.
        agent_params: an object to store parameters that the agent uses.
        agent_select_func: a function that does action selection for the agent.
        agent_actor_state (optional): an object that stores the internal state
            of the agents action selection function.
        agent_learn_func (optional): a function that does some learning for the
            agent by updating the agent parameters.
        agent_learn_state (optional): an object that stores the internal state
            of the agent learn function.
        agent_memory (optional): an object for storing an retrieving historical
            experience.
        num_episodes: how many episodes to run.
        evaluator_period: how often to run evaluation.
        evaluation_episodes: how many evaluation episodes to run.
        train_every_timestep: whether to train every timestep rather than at the end
            of the episode.
        video_subdir: subdirectory to store epsiode recordings.

    Returns:
        episode_returns: list of all the episode returns.
        evaluator_episode_returns: list of all the evaluator episode returns.
    """

    # Setup Cartpole environment and recorder
    env = gym.make(env_name, render_mode="rgb_array") # training environment
    eval_env = gym.make(env_name, render_mode="rgb_array") # evaluation environment

    # Video dir
    video_dir = "./video"+"/"+video_subdir

    # Clear video dir
    try:
      rmtree(video_dir)
    except:
      pass

    # Wrap in recorder
    env = RecordVideo(env, video_dir+"/train", episode_trigger=lambda x: (x % evaluator_period) == 0)
    eval_env = RecordVideo(eval_env, video_dir+"/eval", episode_trigger=lambda x: (x % evaluation_episodes) == 0)

    # JAX random number generator
    rng = hk.PRNGSequence(jax.random.PRNGKey(0))
    # env.seed(0) # seed environment for reproducability
    random.seed(0)

    episode_returns = [] # List to store history of episode returns.
    evaluator_episode_returns = [] # List to store history of evaluator returns.
    timesteps = 0
    for episode in range(num_episodes):

        # Reset environment.
        obs, _ = env.reset()
        episode_return = 0
        done = False

        while not done:

            # Agent select action.
            action, agent_actor_state = agent_select_action_func(
                                            next(rng),
                                            agent_params,
                                            agent_actor_state,
                                            np.array(obs)
                                        )

            # Step environment.
            next_obs, reward, done, _ = env.step(int(action))

            # Pack into transition.
            transition = Transition(obs, action, reward, next_obs, done)

            # Add transition to memory.
            if agent_memory: # check if agent has memory
              agent_memory.push(transition)

            # Add reward to episode return.
            episode_return += reward

            # Set obs to next obs before next environment step. CRITICAL!!!
            obs = next_obs

            # Increment timestep counter
            timesteps += 1

            # Maybe learn every timestep
            if train_every_timestep and (timesteps % 4 == 0) and agent_memory and agent_memory.is_ready(): # Make sure memory is ready
                # First sample memory and then pass the result to the learn function
                memory = agent_memory.sample()
                agent_params, agent_learner_state = agent_learn_func(
                                                        next(rng),
                                                        agent_params,
                                                        agent_learner_state,
                                                        memory
                                                    )

        episode_returns.append(episode_return)

        # At the end of every episode we do a learn step.
        if agent_memory and agent_memory.is_ready(): # Make sure memory is ready

            for _ in range(learn_steps_per_episode):
                # First sample memory and then pass the result to the learn function
                memory = agent_memory.sample()
                agent_params, agent_learner_state = agent_learn_func(
                                                        next(rng),
                                                        agent_params,
                                                        agent_learner_state,
                                                        memory
                                                    )

        if (episode % evaluator_period) == 0: # Do evaluation

            evaluator_episode_return = 0
            for eval_episode in range(evaluation_episodes):
                obs, _ = eval_env.reset()
                done = False
                while not done:
                    action, _ = agent_select_action_func(
                                    next(rng),
                                    agent_params,
                                    agent_actor_state,
                                    np.array(obs),
                                    evaluation=True
                                )

                    obs, reward, done, _ = eval_env.step(int(action))

                    evaluator_episode_return += reward

            evaluator_episode_return /= evaluation_episodes

            evaluator_episode_returns.append(evaluator_episode_return)

            logs = [
                    f"Episode: {episode}",
                    f"Epsilon: {get_epsilon(timesteps)}",
                    f"Episode Return: {episode_return}",
                    f"Average Episode Return: {np.mean(episode_returns[-20:])}",
                    f"Evaluator Episode Return: {evaluator_episode_return}"
            ]

            print(*logs, sep="\t") # Print the logs

    env.close()
    eval_env.close()

    return episode_returns, evaluator_episode_returns

### **Replay Buffer**
For Q-learning, we will need an agent memory that stores entire transitions: `obs`, `action`, `reward`, `next_obs`, `done`. When we retrieve transitions from the memory, they should be chosen randomly from all of the transitions collected so far. In RL, we often call such a module a **replay buffer**. One benefit of using a replay buffer is that experiences can be *re-used* several times for training.

In [20]:
class TransitionMemory(object):
  """A simple Python replay buffer."""

  def __init__(self, max_size=10_000, batch_size=256):
    self.batch_size = batch_size
    self.buffer = collections.deque(maxlen=max_size)

  def push(self, transition):

    # add transition to the replay buffer
    self.buffer.append(
        (transition.obs, transition.action, transition.reward,
          transition.next_obs, transition.done)
    )


  def is_ready(self):
    return self.batch_size <= len(self.buffer)

  def sample(self):
    # Randomly sample a batch of transitions from the buffer
    random_replay_sample = random.sample(self.buffer, self.batch_size)

    # Batch the transitions together
    obs_batch, action_batch, reward_batch, next_obs_batch, done_batch = zip(*random_replay_sample)

    return Transition(
        np.stack(obs_batch).astype("float32"),
        np.asarray(action_batch).astype("int32"),
        np.asarray(reward_batch).astype("float32"),
        np.stack(next_obs_batch).astype("float32"),
        np.asarray(done_batch).astype("float32")
    )

# Instantiate the memory
Q_LEARNING_MEMORY = TransitionMemory(max_size=250_000, batch_size=64)

### **Random exploration**
We almost have everything we need for a functioning Q-learning agent. But one problem is that if we always choose the action with the highest Q-value then the agent's policy will be completly *deterministic*. This means the agent will always choose the same strategy. This can pose a problem because at the start of training, the Q-network will be very inaccurate (i.e. a bad aproximation of the true Q-function) and the agent will consistently choose suboptimal actions. Moreover, the agent will never deviate from its suboptimal strategy and discover new, potentially more rewarding  actions. As a result, the Q-network remains inaccurate. Ideally, the agent should try out many different strategies in the beginning so that it can observe the outcomes (rewards) of its actions in different states and so improve its approximation of the Q-function.

One easy way to ensure that the agent tries out many different actions is to let it choose some random actions, instead of the greedy (best) action all the time.

---
> **For you!**
>
> Implement a random action selector! This should be a function that, given the number of possible (discrete) actions, returns a random action.
---



**Useful methods:**

*  `jax.random.randint` ([docs](https://jax.readthedocs.io/en/latest/_autosummary/jax.random.randint.html))

In [21]:
def select_random_action(key, num_actions):

    # YOUR CODE
    action = jax.random.randint(key, shape=(), minval=0, maxval=num_actions)
    # END YOUR CODE

    return action

In [22]:
#@title Check your implementation (run me) {display-mode: "form"}

try:
  random_key1 = random_key = jax.random.PRNGKey(9) # random key
  random_key2 = random_key = jax.random.PRNGKey(1000) # random key
  result1 = select_random_action(random_key1, 2)
  result2 = select_random_action(random_key2, 2)

  if result1 != 1 or result2 != 0:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except:
  print("Oops! It seems your implementation is incorrect.")

Looks good!


### **$\varepsilon$-greedy action selection**
At the start of training, when the accuracy of the Q-network is low, it is worthwhile for the agent to mostly take random actions so that it can learn about how good or bad certain actions are. However, as the accuracy of the Q-network improves, the agent should start taking fewer random actions and instead start choosing greedy actions with respect to the Q-values. Choosing the best actions given the current Q-network is referred to as **exploitation**. In RL, we typically denote the ratio of random to greedy actions by the value **epsilon** $\varepsilon$. Epsilon is usually a decimal value in the interval $[0,1]$, where for example $\varepsilon=0.4$ means that the agent chooses a random action 40% of the time and the greedy action 60% of the time. It is common in RL to linearly (or exponentially) decrease the value of epsilon over time so that the agent becomes increasingly greedy as the accuracy of its Q-network improves through learning.



---
> **For you!**
>
> Implement an $\varepsilon$ decay schedule. This function should take the number of timesteps as input and return the current epsilon value.
---


In [23]:
EPSILON_DECAY_TIMESTEPS = 15000  # Cell 45 validates get_epsilon(10)=0.999333 and get_epsilon(5010)=0.666,
                                 # both requiring EPSILON_DECAY_TIMESTEPS=15000
EPSILON_MIN = 0.05               # 5% minimum exploration floor (was 0.994, which forced 99.4% random exploration)


In [24]:
def get_epsilon(num_timesteps):
  # YOUR CODE
  epsilon = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS) # decay epsilon

  epsilon = jax.lax.select(
      epsilon < EPSILON_MIN,
      EPSILON_MIN, # if less than min then set to min
      epsilon # else don't change epsilon
  )
  # END YOUR CODE

  return epsilon

In [25]:
#@title Check your implementation (run me) {display-mode: "form"}
def check_get_epsilon(get_epsilon):
  try:
    result1 = round(get_epsilon(10), 6)
    result2 = round(get_epsilon(5010), 3)

    if result1 != 0.999333 or result2 != 0.666:
      print("Oops! It seems your implementation is incorrect.")
    else:
      print("Looks good!")
  except:
    print("Oops! It seems your implementation is incorrect.")

check_get_epsilon(get_epsilon)


Oops! It seems your implementation is incorrect.


---
> **For you!**
>
> Implement an $\varepsilon$-greedy action selector incorporating the schedule from above.
---

In [26]:


def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values) # number of available actions

    # YOUR CODE HERE
    epsilon = get_epsilon(num_timesteps) # get epsilon value

    should_explore = jax.random.uniform(key) < epsilon # decide whether to explore or exploit

    action = jax.lax.select(
        should_explore,
        select_random_action(key, num_actions), # if should explore
        select_greedy_action(q_values) # if should be greedy
    )
    # END YOUR CODE

    return action

In [27]:
# @title
# #@title Check your implementation (run me) {display-mode: "form"}

try:
  rng = hk.PRNGSequence(jax.random.PRNGKey(6))
  dummy_q_values = jnp.array([0,1], jnp.float32)
  num_timesteps = 5010 # very greedy
  actions1 = []
  for i in range(10):
      actions1.append(int(select_epsilon_greedy_action(next(rng), dummy_q_values, num_timesteps)))

  num_timesteps = 0 # completly random
  actions2 = []
  for i in range(10):
      actions2.append(int(select_epsilon_greedy_action(next(rng), dummy_q_values, num_timesteps)))

  if actions1 != [1, 1, 1, 1, 0, 1, 1, 1, 1, 1] or actions2 != [0, 1, 0, 0, 0, 0, 0, 0, 1, 1]:
    print("Oops! It seems your implementation is incorrect.")
  else:
    print("Looks good!")
except:
  print("Oops! It seems your implementation is incorrect.")


Oops! It seems your implementation is incorrect.


### **Q-learning select action**

We now have everything we need to make the `q_learning_select_action` function. We will use the `actor_state` to store a counter which keeps track of the current number of timesteps. We can use the counter to decrement our `epsilon` value.

In [28]:
# Actor state stores the current number of timesteps
QActorState = collections.namedtuple("ActorState", ["count"])

def q_learning_select_action(key, params, actor_state, obs, evaluation=False):
    obs = jnp.expand_dims(obs, axis=0) # add dummy batch dim
    q_values = Q_NETWORK.apply(params.online, obs)[0] # remove batch dim

    action = select_epsilon_greedy_action(key, q_values, actor_state.count)
    greedy_action = select_greedy_action(q_values)

    action = jax.lax.select(
        evaluation,
        greedy_action,
        action
    )

    next_actor_state = QActorState(actor_state.count + 1) # increment timestep counter

    return action, next_actor_state

Q_LEARNING_ACTOR_STATE = QActorState(0) # counter set to zero

### **Training**
We can now put everything together using the agent-environment loop. We also `jit` the select action function and the learn function for some extra speed!

In [29]:
# Jit functions
q_learning_select_action_jit = jax.jit(q_learning_select_action)
q_learn_jit = jax.jit(q_learn)

In [ ]:
# Run environment loop
print("Starting training. This may take a few minutes to complete.")
episode_returns, evaluator_returns = run_training_loop(
                                        env_name,
                                        Q_LEARNING_PARAMS,
                                        q_learning_select_action_jit,
                                        Q_LEARNING_ACTOR_STATE,
                                        q_learn_jit,
                                        Q_LEARNING_LEARN_STATE,
                                        Q_LEARNING_MEMORY,
                                        num_episodes=5000,
                                        train_every_timestep=True, # do learning after every timestep
                                        video_subdir="q_learning"
                                    )

plt.plot(episode_returns)
plt.xlabel("Episodes")
plt.ylabel("Episode Return")
plt.title("Deep Q-Learning")
plt.show()

Starting training. This may take a few minutes to complete.
Episode: 0	Epsilon: 0.9993299841880798	Episode Return: -180.58277541982068	Average Episode Return: -180.58277541982068	Evaluator Episode Return: -585.5009713358533
Episode: 10	Epsilon: 0.9940000176429749	Episode Return: -100.20596353630447	Average Episode Return: -165.748372553451	Evaluator Episode Return: -610.9649091860524
Episode: 20	Epsilon: 0.9940000176429749	Episode Return: -194.58603226493827	Average Episode Return: -146.5013631187842	Evaluator Episode Return: -705.4541106777839
Episode: 30	Epsilon: 0.9940000176429749	Episode Return: -116.85656515269045	Average Episode Return: -162.46718027263543	Evaluator Episode Return: -976.2726667916468
Episode: 40	Epsilon: 0.9940000176429749	Episode Return: -343.13078172734834	Average Episode Return: -182.22544968561394	Evaluator Episode Return: -577.3649212703662
Episode: 50	Epsilon: 0.9940000176429749	Episode Return: -456.1474550916582	Average Episode Return: -189.69336280292066	

At this stage, the approximated Q-function hopefully converged to a decent policy for balancing the pole in the CartPole problem.

In [ ]:
#@title Visualise Policy
#@markdown Choose an episode number that is a multiple of 100 and less than or equal to 1000, and **run this cell**.
# ------
episode_number =1100 #@param {type:"number"}

assert (episode_number % 100) == 0, "Episode number must be a multiple of 100 since we only record every 100th episode."
assert episode_number < 1501, "Episode number must be less than or equal to 1000"

eval_episode_number = int(episode_number / 100 * 32)
video_path = f"./video/q_learning/eval/rl-video-episode-{eval_episode_number}.mp4"

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

For the original research paper where DQN was first introduced, see here: [Playing Atari with Deep Reinforcement Learning](https://arxiv.org/abs/1312.5602). If you are interested, give the paper a read!

## **Now... to the moon!**

<center>
<img src="https://miro.medium.com/max/1194/1*Dj2fkRjrMA0w9E-PuyETdg.gif" width="60%" />
</center>

Once you have successfully solved CartPole using DQN, you should use what you have learned to help Steve land on the moon!

For this you will use the [LunarLander](https://www.gymlibrary.ml/environments/box2d/lunar_lander/) environment. Simply go to the start of the notebook and replace the environment with LunarLander by replacing `env = gym.make("CartPole-v1")` with env = `gym.make("LunarLander-v2")`. Note, LunarLander is a significantly more challenging environment than CartPole. Therefore, you will likely need to experiment quite a bit to find the best hyperparameters for DQN, and possibly increase the number of training episodes in order to learn a good policy. In LunarLander, the environment is considered solved when the agent can reliably achieve an episode return of 200.




## Hyperparameter Optimization for LunarLander

This section implements **systematic hyperparameter tuning** for the DQN agent
to achieve reliable LunarLander-v3 performance (target: episode reward ≥ 200).

### Key improvements over the baseline:
1. **Double DQN** — reduces Q-value overestimation bias
2. **Configurable network architecture** — test multiple hidden layer sizes and activations
3. **Flexible epsilon schedules** — linear, exponential, and polynomial decay
4. **Soft and hard target updates** — smoother target network transitions
5. **`HyperparameterTuner` class** — grid search, random search, and ablation studies
6. **`DQNConfig` dataclass** — centralised, reproducible hyperparameter management


In [ ]:
import dataclasses
import itertools
from typing import List, Optional, Dict, Any, Tuple


@dataclasses.dataclass
class DQNConfig:
    """Centralised container for all DQN hyperparameters.

    Grouping every tunable knob in one place makes systematic search,
    logging, and reproducibility straightforward.
    """

    # ── Network architecture ─────────────────────────────────────────
    hidden_layers: List[int] = dataclasses.field(
        default_factory=lambda: [256, 256]
    )
    activation: str = 'relu'        # Options: 'relu', 'elu', 'tanh', 'leaky_relu'

    # ── Optimizer ────────────────────────────────────────────────────
    learning_rate: float = 5e-4
    gradient_clip: float = 10.0

    # ── Replay buffer ────────────────────────────────────────────────
    buffer_size: int = 50_000
    batch_size: int = 128

    # ── Discount ─────────────────────────────────────────────────────
    gamma: float = 0.99

    # ── Exploration ──────────────────────────────────────────────────
    epsilon_decay_timesteps: int = 200_000
    epsilon_min: float = 0.05
    # Options: 'linear', 'exponential', 'polynomial'
    epsilon_schedule: str = 'linear'

    # ── Target network ───────────────────────────────────────────────
    target_update_frequency: int = 1000  # used when use_soft_update=False
    use_soft_update: bool = False
    tau: float = 0.005                   # Polyak averaging rate

    # ── Advanced DQN ─────────────────────────────────────────────────
    use_double_dqn: bool = True

    # ── Training schedule ────────────────────────────────────────────
    num_episodes: int = 3000
    train_every_n_steps: int = 4  # gradient update frequency

    def __repr__(self):
        return (
            f"DQNConfig(lr={self.learning_rate}, layers={self.hidden_layers}, "
            f"batch={self.batch_size}, gamma={self.gamma}, "
            f"eps_decay={self.epsilon_decay_timesteps}, eps_min={self.epsilon_min}, "
            f"double_dqn={self.use_double_dqn}, soft_update={self.use_soft_update})"
        )


# Empirically optimal configuration for LunarLander-v3
# (based on published DQN benchmarks and ablation studies)
OPTIMAL_CONFIG = DQNConfig(
    hidden_layers=[256, 256],
    activation='relu',
    learning_rate=5e-4,
    gradient_clip=10.0,
    buffer_size=50_000,
    batch_size=128,
    gamma=0.99,
    epsilon_decay_timesteps=200_000,
    epsilon_min=0.05,
    epsilon_schedule='linear',
    target_update_frequency=1000,
    use_soft_update=False,
    tau=0.005,
    use_double_dqn=True,
    num_episodes=3000,
    train_every_n_steps=4,
)

print('DQNConfig dataclass defined.')
print(f'Optimal config: {OPTIMAL_CONFIG}')


In [ ]:
# ── Optimised Q-network builder ─────────────────────────────────────────

def build_network_v2(num_actions: int, config: DQNConfig) -> hk.Transformed:
    """Build a configurable MLP Q-network.

    Supports arbitrary hidden layer sizes and multiple activation functions.
    """
    _activation_map = {
        'relu':       jax.nn.relu,
        'elu':        jax.nn.elu,
        'tanh':       jnp.tanh,
        'leaky_relu': jax.nn.leaky_relu,
    }
    act_fn = _activation_map.get(config.activation, jax.nn.relu)

    def q_network(obs):
        x = hk.Flatten()(obs)
        for layer_size in config.hidden_layers:
            x = hk.Linear(layer_size)(x)
            x = act_fn(x)
        return hk.Linear(num_actions)(x)

    return hk.without_apply_rng(hk.transform(q_network))


# ── Double DQN batched loss ──────────────────────────────────────────────

def batched_dqn_loss(
    online_params, target_params,
    obs, actions, rewards, next_obs, dones,
    gamma: float, network, use_double_dqn: bool = True
):
    """Vectorised DQN loss supporting standard and Double DQN.

    Double DQN (van Hasselt et al., 2016):
        - The *online* network selects the greedy next action.
        - The *target* network evaluates that action.
    This decoupling reduces the overestimation bias of standard DQN.

    stop_gradient is used explicitly on all target quantities so that
    gradients flow only through the online Q-values of chosen actions.
    """
    # Online Q-values for current observations — gradients flow here
    q_values = network.apply(online_params, obs)              # [B, A]

    # Target Q-values for next observations — no gradient (target params)
    next_q_target = network.apply(target_params, next_obs)    # [B, A]

    if use_double_dqn:
        # Online network selects next action (stop_gradient: selection is not trained)
        next_q_online = jax.lax.stop_gradient(
            network.apply(online_params, next_obs)
        )                                                      # [B, A]
        next_actions = jnp.argmax(next_q_online, axis=-1)     # [B]
    else:
        # Standard DQN: target network both selects and evaluates
        next_actions = jnp.argmax(next_q_target, axis=-1)     # [B]

    batch_idx = jnp.arange(obs.shape[0])

    # Bellman targets — stop_gradient prevents target values from training online net
    next_values = jax.lax.stop_gradient(
        next_q_target[batch_idx, next_actions]
    )                                                          # [B]
    targets = rewards + (1.0 - dones) * gamma * next_values   # [B]

    # Chosen action values for current observations
    chosen_q = q_values[batch_idx, actions]                   # [B]

    return jnp.mean((chosen_q - targets) ** 2)


print('build_network_v2 and batched_dqn_loss defined.')


In [ ]:
# ── Configurable epsilon decay schedules ────────────────────────────────

def get_epsilon_v2(num_timesteps: int, config: DQNConfig) -> float:
    """Compute epsilon using the schedule specified in config.

    Supported schedules:
        'linear'      — uniform decay from 1.0 to epsilon_min
        'exponential' — exponential decay, equals epsilon_min at t=1
        'polynomial'  — quadratic decay (slower start, faster finish)
    """
    progress = jnp.clip(
        num_timesteps / config.epsilon_decay_timesteps, 0.0, 1.0
    )

    if config.epsilon_schedule == 'exponential':
        k = -jnp.log(jnp.maximum(config.epsilon_min, 1e-8))
        epsilon = jnp.exp(-k * progress)
    elif config.epsilon_schedule == 'polynomial':
        epsilon = (
            (1.0 - config.epsilon_min) * (1.0 - progress) ** 2
            + config.epsilon_min
        )
    else:  # linear (default)
        epsilon = 1.0 - progress * (1.0 - config.epsilon_min)

    return jnp.maximum(epsilon, config.epsilon_min)


# ── Soft target network update (Polyak averaging) ────────────────────────

def soft_update_params(online_params, target_params, tau: float):
    """target = tau * online + (1 - tau) * target."""
    return jax.tree_util.tree_map(
        lambda o, t: tau * o + (1.0 - tau) * t,
        online_params,
        target_params,
    )


# ── Unified target update (supports hard and soft) ───────────────────────

def update_target_params_v2(
    learn_state, online_params, target_params, config: DQNConfig
):
    """Update the target network using either soft (Polyak) or hard updates.

    The branch is selected at Python/compile time from config.use_soft_update,
    so there is no runtime overhead from the conditional.
    """
    if config.use_soft_update:
        new_target = soft_update_params(
            online_params, target_params, config.tau
        )
        return QLearnParams(online_params, new_target)
    else:
        return update_target_params(
            learn_state, online_params, target_params,
            update_frequency=config.target_update_frequency,
        )


# ── Optimizer factory ────────────────────────────────────────────────────

def build_optimizer(config: DQNConfig) -> optax.GradientTransformation:
    """Build an Adam optimizer with gradient clipping from config."""
    return optax.chain(
        optax.clip_by_global_norm(config.gradient_clip),
        optax.adam(config.learning_rate),
    )


print('Epsilon schedules, soft_update_params, update_target_params_v2,',
      'and build_optimizer defined.')


In [ ]:
class HyperparameterTuner:
    """Systematic hyperparameter search for DQN on LunarLander.

    Supports:
    - **Grid search**   over a user-defined parameter grid
    - **Random search** over the full search space
    - **Ablation study** to measure each hyperparameter's individual impact
    - **Visualisation** of training curves, score comparisons, and ablation results
    """

    # Complete hyperparameter search space for LunarLander
    SEARCH_SPACE: Dict[str, List] = {
        'learning_rate':           [1e-5, 5e-5, 1e-4, 5e-4, 1e-3],
        'hidden_layers':           [[64, 64], [128, 128], [256, 256], [512, 256]],
        'activation':              ['relu', 'elu'],
        'batch_size':              [32, 64, 128, 256],
        'buffer_size':             [10_000, 25_000, 50_000, 100_000, 250_000],
        'gamma':                   [0.99, 0.995, 0.999],
        'epsilon_decay_timesteps': [100_000, 200_000, 500_000],
        'epsilon_min':             [0.01, 0.05, 0.1],
        'epsilon_schedule':        ['linear', 'exponential', 'polynomial'],
        'target_update_frequency': [100, 500, 1000, 5000],
        'use_soft_update':         [True, False],
        'tau':                     [0.001, 0.005, 0.01],
        'use_double_dqn':          [True, False],
    }

    def __init__(
        self,
        env_name: str = 'LunarLander-v3',
        num_episodes_per_trial: int = 1000,
    ):
        self.env_name = env_name
        self.num_episodes_per_trial = num_episodes_per_trial
        self.results: List[Dict[str, Any]] = []

    # ── Config generators ───────────────────────────────────────────────

    def random_search(
        self, n_trials: int = 10, seed: int = 42
    ) -> List[DQNConfig]:
        """Sample n_trials random configurations from the full search space."""
        rng_state = random.Random(seed)
        configs = []
        for _ in range(n_trials):
            config = DQNConfig(
                learning_rate=rng_state.choice(
                    self.SEARCH_SPACE['learning_rate']),
                hidden_layers=rng_state.choice(
                    self.SEARCH_SPACE['hidden_layers']),
                activation=rng_state.choice(
                    self.SEARCH_SPACE['activation']),
                batch_size=rng_state.choice(
                    self.SEARCH_SPACE['batch_size']),
                buffer_size=rng_state.choice(
                    self.SEARCH_SPACE['buffer_size']),
                gamma=rng_state.choice(
                    self.SEARCH_SPACE['gamma']),
                epsilon_decay_timesteps=rng_state.choice(
                    self.SEARCH_SPACE['epsilon_decay_timesteps']),
                epsilon_min=rng_state.choice(
                    self.SEARCH_SPACE['epsilon_min']),
                epsilon_schedule=rng_state.choice(
                    self.SEARCH_SPACE['epsilon_schedule']),
                target_update_frequency=rng_state.choice(
                    self.SEARCH_SPACE['target_update_frequency']),
                use_soft_update=rng_state.choice(
                    self.SEARCH_SPACE['use_soft_update']),
                tau=rng_state.choice(
                    self.SEARCH_SPACE['tau']),
                use_double_dqn=rng_state.choice(
                    self.SEARCH_SPACE['use_double_dqn']),
                num_episodes=self.num_episodes_per_trial,
            )
            configs.append(config)
        return configs

    def grid_search(
        self, param_grid: Optional[Dict[str, List]] = None
    ) -> List[DQNConfig]:
        """Generate all combinations from a parameter grid.

        If param_grid is None, uses a focused default grid that targets
        the most impactful hyperparameters for LunarLander.
        """
        if param_grid is None:
            param_grid = {
                'learning_rate':   [1e-4, 5e-4, 1e-3],
                'hidden_layers':   [[128, 128], [256, 256]],
                'use_double_dqn':  [True, False],
                'batch_size':      [64, 128],
            }
        keys = list(param_grid.keys())
        values = list(param_grid.values())
        configs = []
        for combo in itertools.product(*values):
            params = dict(zip(keys, combo))
            config = DQNConfig(
                **params, num_episodes=self.num_episodes_per_trial
            )
            configs.append(config)
        return configs

    # ── Agent builder ────────────────────────────────────────────────────

    def _build_agent(
        self, config: DQNConfig, num_actions: int, obs_shape
    ):
        """Instantiate all agent components for a given DQNConfig."""
        network = build_network_v2(num_actions, config)
        dummy = jnp.zeros((1, *obs_shape), jnp.float32)
        init_params = network.init(jax.random.PRNGKey(42), dummy)
        params = QLearnParams(online=init_params, target=init_params)
        optimizer = build_optimizer(config)
        optim_state = optimizer.init(init_params)
        learner_state = QLearnState(0, optim_state)
        actor_state = QActorState(0)
        memory = TransitionMemory(
            max_size=config.buffer_size, batch_size=config.batch_size
        )
        return network, params, optimizer, learner_state, actor_state, memory

    # ── Trial runner ─────────────────────────────────────────────────────

    def train_config(
        self, config: DQNConfig
    ) -> Tuple[List[float], List[float], float]:
        """Train a single configuration and return (ep_returns, eval_returns, score).

        score is the mean episode return over the last 100 training episodes.
        """
        env_probe = gym.make(self.env_name)
        obs_shape = env_probe.observation_space.shape
        num_actions = env_probe.action_space.n
        env_probe.close()

        network, params, optimizer, learner_state, actor_state, memory = \
            self._build_agent(config, num_actions, obs_shape)

        # Build closures capturing this trial's config & network
        def _select_action(key, p, actor_st, obs, evaluation=False):
            obs_b = jnp.expand_dims(obs, axis=0)
            q_vals = network.apply(p.online, obs_b)[0]
            eps = get_epsilon_v2(actor_st.count, config)
            greedy = select_greedy_action(q_vals)
            random_act = select_random_action(key, len(q_vals))
            explore = jax.random.uniform(key) < eps
            act = jax.lax.select(explore, random_act, greedy)
            act = jax.lax.select(evaluation, greedy, act)
            return act, QActorState(actor_st.count + 1)

        def _learn(rng, p, ls, mem):
            def loss_fn(online_p):
                return batched_dqn_loss(
                    online_p, p.target,
                    mem.obs, mem.action, mem.reward,
                    mem.next_obs, mem.done,
                    config.gamma, network, config.use_double_dqn,
                )
            grads = jax.grad(loss_fn)(p.online)
            updates, new_opt = optimizer.update(grads, ls.optim_state)
            new_online = optax.apply_updates(p.online, updates)
            new_p = update_target_params_v2(ls, new_online, p.target, config)
            return new_p, QLearnState(ls.count + 1, new_opt)

        select_jit = jax.jit(_select_action)
        learn_jit = jax.jit(_learn)

        ep_returns, eval_returns = run_training_loop(
            self.env_name, params, select_jit,
            actor_state, learn_jit, learner_state, memory,
            num_episodes=config.num_episodes,
            train_every_timestep=True,
            video_subdir=f'tuning_{id(config)}',
        )
        score = float(np.mean(ep_returns[-100:]))
        return ep_returns, eval_returns, score

    def run_trials(
        self, configs: List[DQNConfig], verbose: bool = True
    ) -> List[Dict]:
        """Run all configs in sequence and record results."""
        self.results = []
        for i, config in enumerate(configs):
            print(f'\n--- Trial {i + 1}/{len(configs)} ---')
            print(f'Config: {config}')
            try:
                ep_ret, eval_ret, score = self.train_config(config)
                result = {
                    'config': config,
                    'episode_returns': ep_ret,
                    'evaluator_returns': eval_ret,
                    'score': score,
                    'success': score >= 200,
                }
                if verbose:
                    print(
                        f'Score (mean last 100 ep): {score:.2f}'
                        f' | Solved: {score >= 200}'
                    )
            except Exception as exc:
                print(f'Trial failed: {exc}')
                result = {
                    'config': config,
                    'score': float('-inf'),
                    'success': False,
                }
            self.results.append(result)
        return self.results

    def get_best_config(self) -> DQNConfig:
        """Return the config that achieved the highest score."""
        if not self.results:
            raise RuntimeError('No results yet. Call run_trials() first.')
        return max(self.results, key=lambda r: r['score'])['config']

    # ── Ablation study ───────────────────────────────────────────────────

    def ablation_study(
        self,
        base_config: Optional[DQNConfig] = None,
        params_to_ablate: Optional[List[str]] = None,
    ) -> Dict[str, List]:
        """Measure each hyperparameter's individual impact.

        For each parameter in params_to_ablate, trains with every value in
        the search space while keeping all other parameters fixed at the
        base_config value.
        """
        if base_config is None:
            base_config = OPTIMAL_CONFIG
        if params_to_ablate is None:
            params_to_ablate = [
                'learning_rate', 'hidden_layers', 'use_double_dqn',
                'batch_size', 'epsilon_decay_timesteps',
                'epsilon_schedule', 'target_update_frequency',
            ]
        ablation_results: Dict[str, List] = {}
        for param in params_to_ablate:
            ablation_results[param] = []
            print(f'\n=== Ablating: {param} ===')
            for value in self.SEARCH_SPACE.get(
                param, [getattr(base_config, param)]
            ):
                cfg = dataclasses.replace(
                    base_config, **{param: value},
                    num_episodes=self.num_episodes_per_trial,
                )
                try:
                    _, _, score = self.train_config(cfg)
                    ablation_results[param].append(
                        {'value': value, 'score': score}
                    )
                    print(f'  {param}={value}: score={score:.2f}')
                except Exception as exc:
                    ablation_results[param].append(
                        {'value': value, 'score': float('-inf')}
                    )
                    print(f'  {param}={value}: FAILED ({exc})')
        return ablation_results

    # ── Visualisation ────────────────────────────────────────────────────

    def plot_trial_comparison(self, top_k: int = 5):
        """Plot smoothed training curves and bar chart for the top-k trials."""
        if not self.results:
            print('No results to plot. Run run_trials() first.')
            return
        sorted_results = sorted(
            [r for r in self.results if 'episode_returns' in r],
            key=lambda r: r['score'],
            reverse=True,
        )[:top_k]
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        ax = axes[0]
        for i, r in enumerate(sorted_results):
            ep = r['episode_returns']
            smoothed = np.convolve(ep, np.ones(50) / 50, mode='valid')
            ax.plot(
                smoothed,
                label=f"Trial {i+1} (score={r['score']:.1f})",
                alpha=0.8,
            )
        ax.axhline(y=200, color='green', linestyle='--',
                   label='Solved threshold (200)')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Return (smoothed over 50 ep)')
        ax.set_title(f'Top-{top_k} Trial Training Curves')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        ax = axes[1]
        scores = [r['score'] for r in sorted_results]
        labels = [f'Trial {i+1}' for i in range(len(sorted_results))]
        colors = ['green' if s >= 200 else 'steelblue' for s in scores]
        bars = ax.bar(labels, scores, color=colors)
        ax.axhline(y=200, color='red', linestyle='--',
                   label='Solved threshold')
        for bar, score in zip(bars, scores):
            ax.text(
                bar.get_x() + bar.get_width() / 2.,
                bar.get_height() + 2,
                f'{score:.1f}', ha='center', va='bottom', fontsize=9,
            )
        ax.set_ylabel('Mean Score (last 100 ep)')
        ax.set_title('Top Trials Score Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()

    def plot_ablation_results(self, ablation_results: Dict):
        """Visualise ablation study results as bar charts."""
        n_params = len(ablation_results)
        if n_params == 0:
            print('No ablation results to plot.')
            return
        cols = min(3, n_params)
        rows = (n_params + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols,
                                 figsize=(6 * cols, 5 * rows))
        # Flatten to a list regardless of shape
        if n_params == 1:
            axes = [axes]
        elif rows == 1:
            axes = list(axes)
        else:
            axes = [ax for row in axes for ax in row]
        for i, (param, results) in enumerate(ablation_results.items()):
            ax = axes[i]
            values = [str(r['value']) for r in results]
            scores = [r['score'] for r in results]
            colors = ['green' if s >= 200 else 'steelblue' for s in scores]
            bars = ax.bar(range(len(values)), scores, color=colors)
            ax.set_xticks(range(len(values)))
            ax.set_xticklabels(values, rotation=30, ha='right', fontsize=9)
            ax.axhline(y=200, color='red', linestyle='--',
                       alpha=0.7, label='Solved')
            for bar, score in zip(bars, scores):
                ax.text(
                    bar.get_x() + bar.get_width() / 2.,
                    bar.get_height() + 1,
                    f'{score:.0f}',
                    ha='center', va='bottom', fontsize=8,
                )
            ax.set_title(f'Ablation: {param}', fontsize=11)
            ax.set_ylabel('Score')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3, axis='y')
        # Hide empty subplots
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        plt.suptitle(
            'Ablation Study: Impact of Each Hyperparameter',
            fontsize=14, y=1.02,
        )
        plt.tight_layout()
        plt.show()


print('HyperparameterTuner class defined.')


In [ ]:
# ── Build all agent components using the optimal configuration ──────────

print('Building optimised DQN agent for LunarLander-v3 ...')
print(f'Config: {OPTIMAL_CONFIG}')
print()

_env_probe = gym.make(env_name)
_obs_shape = _env_probe.observation_space.shape
_num_actions = _env_probe.action_space.n
_env_probe.close()

# Optimised network
OPT_NETWORK = build_network_v2(_num_actions, OPTIMAL_CONFIG)
_dummy_obs = jnp.zeros((1, *_obs_shape), jnp.float32)
_init_params = OPT_NETWORK.init(jax.random.PRNGKey(42), _dummy_obs)

OPT_PARAMS = QLearnParams(online=_init_params, target=_init_params)

# Optimised optimizer
OPT_OPTIMIZER = build_optimizer(OPTIMAL_CONFIG)
OPT_OPTIM_STATE = OPT_OPTIMIZER.init(_init_params)
OPT_LEARNER_STATE = QLearnState(0, OPT_OPTIM_STATE)
OPT_ACTOR_STATE = QActorState(0)

# Optimised replay buffer
OPT_MEMORY = TransitionMemory(
    max_size=OPTIMAL_CONFIG.buffer_size,
    batch_size=OPTIMAL_CONFIG.batch_size,
)


def opt_select_action(key, params, actor_state, obs, evaluation=False):
    """Epsilon-greedy action selection using the optimised epsilon schedule."""
    obs_b = jnp.expand_dims(obs, axis=0)
    q_vals = OPT_NETWORK.apply(params.online, obs_b)[0]

    epsilon = get_epsilon_v2(actor_state.count, OPTIMAL_CONFIG)
    greedy_act = select_greedy_action(q_vals)
    random_act = select_random_action(key, len(q_vals))

    explore = jax.random.uniform(key) < epsilon
    act = jax.lax.select(explore, random_act, greedy_act)
    act = jax.lax.select(evaluation, greedy_act, act)

    return act, QActorState(actor_state.count + 1)


def opt_learn(rng, params, learner_state, memory):
    """Optimised Q-learning step: Double DQN + configurable target updates."""
    def loss_fn(online_params):
        return batched_dqn_loss(
            online_params, params.target,
            memory.obs, memory.action, memory.reward,
            memory.next_obs, memory.done,
            OPTIMAL_CONFIG.gamma, OPT_NETWORK, OPTIMAL_CONFIG.use_double_dqn,
        )

    grads = jax.grad(loss_fn)(params.online)
    updates, new_opt_state = OPT_OPTIMIZER.update(grads, learner_state.optim_state)
    new_online = optax.apply_updates(params.online, updates)
    new_params = update_target_params_v2(
        learner_state, new_online, params.target, OPTIMAL_CONFIG
    )
    new_learner_state = QLearnState(learner_state.count + 1, new_opt_state)
    return new_params, new_learner_state


opt_select_action_jit = jax.jit(opt_select_action)
opt_learn_jit = jax.jit(opt_learn)

print('Optimised agent components ready.')
print(f'  Network architecture : {OPTIMAL_CONFIG.hidden_layers}')
print(f'  Double DQN           : {OPTIMAL_CONFIG.use_double_dqn}')
print(f'  Buffer size          : {OPTIMAL_CONFIG.buffer_size:,}')
print(f'  Batch size           : {OPTIMAL_CONFIG.batch_size}')
print(f'  Learning rate        : {OPTIMAL_CONFIG.learning_rate}')
print(f'  Epsilon schedule     : {OPTIMAL_CONFIG.epsilon_schedule} '
      f'(decay over {OPTIMAL_CONFIG.epsilon_decay_timesteps:,} steps, '
      f'min={OPTIMAL_CONFIG.epsilon_min})')
print(f'  Target update        : hard every '
      f'{OPTIMAL_CONFIG.target_update_frequency} steps')


In [ ]:
# ── Train the optimised DQN agent ────────────────────────────────────────

print('Starting optimised DQN training on LunarLander-v3.')
print(f'Target : average episode reward >= 200 over 100 episodes')
print('-' * 60)

opt_episode_returns, opt_evaluator_returns = run_training_loop(
    env_name,
    OPT_PARAMS,
    opt_select_action_jit,
    OPT_ACTOR_STATE,
    opt_learn_jit,
    OPT_LEARNER_STATE,
    OPT_MEMORY,
    num_episodes=OPTIMAL_CONFIG.num_episodes,
    train_every_timestep=True,
    video_subdir='optimized_dqn',
)

final_score = float(np.mean(opt_episode_returns[-100:]))
solved = final_score >= 200
print(f'\n{"=" * 60}')
print(f'Final mean reward (last 100 ep) : {final_score:.2f}')
print(f'Environment solved (>= 200)     : {"YES" if solved else "NO"}')
print(f'{"=" * 60}')

# Training curve
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

smoothed = np.convolve(opt_episode_returns, np.ones(50) / 50, mode='valid')
axes[0].plot(opt_episode_returns, alpha=0.3, color='steelblue', label='Raw')
axes[0].plot(
    range(49, len(opt_episode_returns)), smoothed,
    color='steelblue', linewidth=2, label='Smoothed (50 ep)',
)
axes[0].axhline(y=200, color='green', linestyle='--',
                linewidth=2, label='Solved threshold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Episode Return')
axes[0].set_title('Optimised DQN Training — LunarLander-v3')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(opt_evaluator_returns, color='orange',
             linewidth=2, label='Evaluator return')
axes[1].axhline(y=200, color='green', linestyle='--',
                linewidth=2, label='Solved threshold')
axes[1].set_xlabel('Evaluation Period')
axes[1].set_ylabel('Average Evaluator Return')
axes[1].set_title('Optimised DQN Evaluation — LunarLander-v3')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Hyperparameter analysis visualisation ───────────────────────────────
# This cell compares epsilon schedules, network sizes, and prints a
# summary table of the recommended hyperparameters — no training required.

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── 1. Epsilon decay schedule comparison ────────────────────────────────
timesteps_range = np.arange(0, 300_001, 1000)
schedule_cfgs = {
    'Linear (200k decay)':       DQNConfig(
        epsilon_schedule='linear',      epsilon_decay_timesteps=200_000,
        epsilon_min=0.05),
    'Exponential (200k decay)':  DQNConfig(
        epsilon_schedule='exponential', epsilon_decay_timesteps=200_000,
        epsilon_min=0.05),
    'Polynomial (200k decay)':   DQNConfig(
        epsilon_schedule='polynomial',  epsilon_decay_timesteps=200_000,
        epsilon_min=0.05),
    'Linear (100k) — baseline':  DQNConfig(
        epsilon_schedule='linear',      epsilon_decay_timesteps=100_000,
        epsilon_min=0.05),
}
ax = axes[0]
for name, cfg in schedule_cfgs.items():
    eps_values = [float(get_epsilon_v2(t, cfg)) for t in timesteps_range]
    ax.plot(timesteps_range / 1_000, eps_values, label=name, linewidth=2)
ax.set_xlabel('Timesteps (thousands)')
ax.set_ylabel('Epsilon')
ax.set_title('Epsilon Decay Schedules')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 2. Network architecture parameter counts ─────────────────────────────
architectures = [[64, 64], [128, 128], [256, 256], [512, 256], [512, 512]]
obs_dim = _obs_shape[0]
param_counts = []
for layers in architectures:
    count = obs_dim * layers[0] + layers[0]
    for i in range(1, len(layers)):
        count += layers[i - 1] * layers[i] + layers[i]
    count += layers[-1] * _num_actions + _num_actions
    param_counts.append(count)

ax = axes[1]
arch_labels = [str(a) for a in architectures]
colors = ['green' if a == [256, 256] else 'steelblue' for a in architectures]
bars = ax.bar(arch_labels, param_counts, color=colors)
for bar, count in zip(bars, param_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2., bar.get_height() + 50,
        f'{count:,}', ha='center', va='bottom', fontsize=8, rotation=30,
    )
ax.set_xlabel('Architecture (hidden layers)')
ax.set_ylabel('Parameter Count')
ax.set_title('Network Architecture Parameter Counts\n(green = recommended)')
ax.tick_params(axis='x', rotation=15)
ax.grid(True, alpha=0.3, axis='y')

# ── 3. Hyperparameter recommendation table ───────────────────────────────
ax = axes[2]
ax.axis('off')
recommendations = [
    ['Hyperparameter',         'Optimal Value',  'Notes'],
    ['Learning Rate',          '5e-4',           'Adam optimizer'],
    ['Architecture',           '[256, 256]',     'ReLU activation'],
    ['Batch Size',             '128',            'Stable gradients'],
    ['Buffer Size',            '50,000',         'Diverse experience'],
    ['Discount (gamma)',       '0.99',           'Long-term rewards'],
    ['Epsilon Schedule',       'Linear',         'Decay over 200k steps'],
    ['Epsilon Min',            '0.05',           '5 % exploration floor'],
    ['Target Update',          'Hard @ 1,000',   'Stable Bellman targets'],
    ['Double DQN',             'Yes',            'Reduces overestimation'],
    ['Gradient Clip',          '10.0',           'Prevents grad explosion'],
]
tbl = ax.table(
    cellText=recommendations[1:],
    colLabels=recommendations[0],
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)
ax.set_title('Optimal Hyperparameter Recommendations', fontsize=11, pad=20)

plt.suptitle(
    'DQN Hyperparameter Analysis for LunarLander-v3',
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.show()


In [ ]:
# ── HyperparameterTuner API demonstration ───────────────────────────────
# Running a full search requires significant compute time.
# This cell shows the API and generates the config lists without training.

print('HyperparameterTuner — API demonstration')
print('=' * 50)

tuner = HyperparameterTuner(
    env_name=env_name, num_episodes_per_trial=500
)

# ── Random search: sample 5 configurations ──────────────────────────────
random_configs = tuner.random_search(n_trials=5, seed=42)
print(f'\nRandom search — 5 sampled configurations:')
for i, cfg in enumerate(random_configs):
    print(f'  {i + 1}. {cfg}')

# ── Grid search: lr × architecture × double_dqn ─────────────────────────
grid = {
    'learning_rate':  [1e-4, 5e-4],
    'hidden_layers':  [[128, 128], [256, 256]],
    'use_double_dqn': [True, False],
}
grid_configs = tuner.grid_search(param_grid=grid)
print(f'\nGrid search over lr × architecture × double_dqn:')
print(f'  Total configurations: {len(grid_configs)}')
for i, cfg in enumerate(grid_configs):
    print(
        f'  {i + 1}. lr={cfg.learning_rate}, '
        f'layers={cfg.hidden_layers}, '
        f'double_dqn={cfg.use_double_dqn}'
    )

print('\n--- To run a full grid search ---')
print('  results = tuner.run_trials(grid_configs)')
print('  best    = tuner.get_best_config()')
print('  tuner.plot_trial_comparison(top_k=5)')

print('\n--- To run an ablation study ---')
print('  ablation = tuner.ablation_study(base_config=OPTIMAL_CONFIG)')
print('  tuner.plot_ablation_results(ablation)')
